# Kaggle GPU Runner for adaptive-fusion-ts2img

Notebook dùng để chạy code từ GitHub trên Kaggle GPU/CUDA.

Cách dùng: bật `Accelerator = GPU`, bật `Internet = On`, chạy các cell kiểm tra, sau đó dùng `Save Version -> Save & Run All` để Kaggle chạy nền trên server.

Kết quả nên lưu trong `/kaggle/working/outputs`.


In [ ]:
from pathlib import Path
import os, subprocess, sys, platform

REPO_URL = 'https://github.com/hoangtnedu/adaptive-fusion-ts2img.git'
BRANCH = 'main'
PROJECT_NAME = 'adaptive-fusion-ts2img'

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
PROJECT_DIR = KAGGLE_WORKING / PROJECT_NAME
OUTPUT_DIR = KAGGLE_WORKING / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Nếu thầy gắn dataset riêng trong Kaggle, sửa DATA_DIR cho đúng thư mục dataset.
DATA_DIR = KAGGLE_INPUT

print('REPO_URL =', REPO_URL)
print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_DIR =', DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)


## 1. Kiểm tra GPU/CUDA

In [ ]:
print('Python:', sys.version)
print('Platform:', platform.platform())
print('\n===== nvidia-smi =====')
!nvidia-smi


In [ ]:
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA version used by PyTorch:', torch.version.cuda)
    print('GPU count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}:', torch.cuda.get_device_name(i))
else:
    print('Không thấy CUDA GPU. Cần bật Accelerator = GPU trong Kaggle Notebook Settings.')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE =', DEVICE)


## 2. Clone hoặc cập nhật GitHub repo

In [ ]:
if not PROJECT_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print('Repo đã tồn tại:', PROJECT_DIR)
    %cd {PROJECT_DIR}
    !git pull

%cd {PROJECT_DIR}
!pwd
!git status
!find . -maxdepth 2 -type f | sort | head -100


## 3. Cài thư viện phụ thuộc

In [ ]:
if Path('requirements.txt').exists():
    !pip install -q -r requirements.txt
else:
    !pip install -q numpy pandas scikit-learn matplotlib scipy scikit-image pyts tqdm pyyaml

# Bổ sung các gói thường dùng cho pipeline 1D -> 2D time series.
!pip install -q pyts scikit-image pyyaml tqdm
print('Hoàn tất cài thư viện.')


## 4. Kiểm tra project và dataset

In [ ]:
print('===== Script/config/notebook trong repo =====')
!find . -maxdepth 3 \( -name 'run*.py' -o -name 'train*.py' -o -name 'main.py' -o -name '*.ipynb' -o -name '*.yaml' -o -name '*.yml' \) | sort

print('\n===== Dataset trong /kaggle/input =====')
!find /kaggle/input -maxdepth 3 -type f | head -100

print('DATA_DIR =', DATA_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)


## 5. Tạo config Kaggle

In [ ]:
import yaml

configs_dir = PROJECT_DIR / 'configs'
configs_dir.mkdir(exist_ok=True)
CONFIG_PATH = configs_dir / 'kaggle.yaml'

kaggle_config = {
    'device': DEVICE,
    'data_dir': str(DATA_DIR),
    'output_dir': str(OUTPUT_DIR),
    'seed': 42,
    'datasets': ['Coffee', 'ECG200', 'GunPoint', 'FordA', 'Wafer'],
    'methods': ['raw_1d', 'gaf', 'mtf', 'rp', 'stft'],
    'image_size': 64,
    'batch_size': 64,
    'epochs': 50,
    'num_workers': 2,
    'save_checkpoints': True,
    'save_figures': True,
}

with open(CONFIG_PATH, 'w', encoding='utf-8') as f:
    yaml.safe_dump(kaggle_config, f, sort_keys=False, allow_unicode=True)

print('Đã tạo:', CONFIG_PATH)
print(CONFIG_PATH.read_text(encoding='utf-8'))


## 6. Smoke test

In [ ]:
import numpy as np, pandas as pd, sklearn, scipy, skimage, pyts, torch
print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('sklearn:', sklearn.__version__)
print('scipy:', scipy.__version__)
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())

candidate_scripts = [
    'run_experiments.py',
    'scripts/run_experiments.py',
    'src/run_experiments.py',
    'main.py',
    'train.py',
]
existing_scripts = [s for s in candidate_scripts if Path(s).exists()]
print('Script tìm thấy:', existing_scripts)
for s in existing_scripts[:3]:
    print(f'\n--- python {s} --help ---')
    !python {s} --help || true


## 7. Chạy thí nghiệm chính

Bỏ comment lệnh phù hợp với repo. Nên chạy thử với epoch nhỏ trước, sau đó tăng lên khi `Save Version -> Save & Run All`.

In [ ]:
# Cách A: nếu repo hỗ trợ run_experiments.py --config
# !python run_experiments.py --config configs/kaggle.yaml

# Cách B: nếu repo chạy bằng main.py hoặc train.py
# !python main.py --config configs/kaggle.yaml
# !python train.py --config configs/kaggle.yaml

# Cách C: nếu code nhận tham số dòng lệnh riêng
# !python run_experiments.py --data_dir /kaggle/input --output_dir /kaggle/working/outputs --device cuda --epochs 50

print('Hãy bỏ comment một lệnh chạy phù hợp sau khi smoke test không lỗi.')


## 8. Kiểm tra và nén kết quả

In [ ]:
print('===== Output files =====')
!find /kaggle/working/outputs -maxdepth 4 -type f | sort | head -200

print('\n===== CSV files =====')
!find /kaggle/working -name '*.csv' -type f | sort

print('\n===== Figure files =====')
!find /kaggle/working \( -name '*.png' -o -name '*.jpg' \) -type f | sort | head -100

!cd /kaggle/working && zip -r kaggle_outputs.zip outputs >/dev/null 2>&1 || true
!ls -lh /kaggle/working/kaggle_outputs.zip || true


In [ ]:
import pandas as pd
csv_files = list(Path('/kaggle/working').rglob('*.csv'))
print('CSV files:')
for i, p in enumerate(csv_files):
    print(i, p)
if csv_files:
    df = pd.read_csv(csv_files[0])
    display(df.head())
    print('Shape:', df.shape)
else:
    print('Chưa có file CSV.')
